# 22 – Memory & Conversation Checkpointing

LangGraph uses a **checkpointer** to persist conversation state across turns:  
- `MemorySaver` (in-process dict) — default in dev/test  
- `PostgresSaver` — for production (Neon-compatible via `DATABASE_URL`)

This notebook tests multi-turn conversations and state persistence.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

## 1. Checkpointer factory — MemorySaver in dev

In [ ]:
from memory.checkpointer import get_checkpointer

checkpointer = get_checkpointer()
print('Checkpointer type:', type(checkpointer).__name__)
print('Uses in-memory dict:', 'Memory' in type(checkpointer).__name__)

## 2. Multi-turn conversation via graph

In [ ]:
from graph.graph import build_graph

graph = build_graph()
THREAD = 'notebook-memory-test'
config = {'configurable': {'thread_id': THREAD}}

# Turn 1
state1 = {
    'query': 'What is the retention GRR?',
    'thread_id': THREAD,
    'data_products': ['retention'],
    'time_range': 'last_30_days',
}
r1 = graph.invoke(state1, config=config)
print('Turn 1:')
print('  intent  :', r1.get('intent'))
print('  summary :', str(r1.get('final_summary', ''))[:100])

In [ ]:
# Turn 2 — follow-up on same thread
state2 = {
    'query': 'What are the open Jira tickets for retention?',
    'thread_id': THREAD,
    'data_products': ['retention'],
    'time_range': 'last_30_days',
}
r2 = graph.invoke(state2, config=config)
print('Turn 2:')
print('  intent  :', r2.get('intent'))
print('  summary :', str(r2.get('final_summary', ''))[:100])

## 3. Retrieve conversation history from checkpointer

In [ ]:
checkpointer = graph.checkpointer

if checkpointer:
    state = checkpointer.get(config)
    if state and state.values:
        history = state.values.get('conversation_history', [])
        print(f'Conversation history entries: {len(history)}')
        for i, turn in enumerate(history[-4:]):
            role = turn.get('role', 'unknown')
            content = str(turn.get('content', ''))[:80]
            print(f'  [{i}] {role}: {content}')
    else:
        print('No state found (MemorySaver may clear between kernel restarts)')
else:
    print('No checkpointer attached')

## 4. Thread isolation — different threads don't share state

In [ ]:
thread_a = 'isolation-test-A'
thread_b = 'isolation-test-B'

# Run on thread A
graph.invoke(
    {'query': 'retention metrics', 'thread_id': thread_a, 'data_products': ['retention']},
    config={'configurable': {'thread_id': thread_a}},
)

# Run on thread B — different state
graph.invoke(
    {'query': 'bookings metrics', 'thread_id': thread_b, 'data_products': ['bookings']},
    config={'configurable': {'thread_id': thread_b}},
)

cp = graph.checkpointer
if cp:
    sa = cp.get({'configurable': {'thread_id': thread_a}})
    sb = cp.get({'configurable': {'thread_id': thread_b}})
    
    query_a = (sa.values if sa else {}).get('query', 'N/A')
    query_b = (sb.values if sb else {}).get('query', 'N/A')
    
    print(f'Thread A last query: {query_a}')
    print(f'Thread B last query: {query_b}')
    print(f'Threads isolated  : {query_a != query_b}')
else:
    print('Isolation verified by design (no shared state in MemorySaver across different thread_ids)')

## 5. Checkpointer with DATABASE_URL — Postgres config

In [ ]:
# Show how PostgresSaver is selected (without actually connecting)
import inspect
from memory import checkpointer as chk_module

src = inspect.getsource(chk_module.get_checkpointer)
print('get_checkpointer source:')
print(src)